# BQ Gap Tracker — Am I on track for Boston?

**One question:** Based on my actual Strava training data, am I on track to hit a Boston Qualifying (BQ) time — and if so, by when?

This notebook is the written argument. Every step below exists to answer that question. Charts that do not change the verdict are omitted.


## Setup and target

Standards are the BAA table in `src/bq_standards.py` (confirmed current as of July 2026). A **safe target** subtracts a configurable buffer (default **6:00**) because recent acceptance cutoffs have run ~4:34–6:51 faster than the published standard.

> **Fill-in check:** age on Boston 2027 race day is set in `src/config.py` (currently a placeholder). Gender is taken from Strava (`F` → Women). Update the age before treating the verdict as final.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

from src.config import CONFIG
from src.bq_standards import resolve_bq_target, format_hms
from src.config import gender_from_strava_sex
import json

athlete = json.loads(Path("../data/athlete.json").read_text())
gender = CONFIG.gender_override or gender_from_strava_sex(athlete.get("sex"))
target = resolve_bq_target(
    CONFIG.age_on_race_day,
    gender,
    safe_buffer_seconds=CONFIG.safe_buffer_seconds,
    course_net_downhill_ft=CONFIG.course_net_downhill_ft,
)
print(f"Athlete: {athlete.get('firstname')} {athlete.get('lastname')} (Strava sex={athlete.get('sex')} → {gender})")
print(f"Age on race day (config): {CONFIG.age_on_race_day} → age group {target.age_group_label}")
print(f"BQ standard:     {format_hms(target.standard_seconds)}")
print(f"Safe target (−{format_hms(target.safe_buffer_seconds)}): {format_hms(target.safe_target_seconds)}")
print(f"Downhill penalty on chosen course: {format_hms(target.downhill_penalty_seconds)} (DQ={target.downhill_disqualified})")
print(f"Horizon: {CONFIG.qualifying_race_date or CONFIG.qualifying_window_end}")


## 0. Data quality

Before modeling: load the local Strava export, filter to runs, parse pace into **seconds per meter**, and **flag** missing HR / implausible paces instead of silently dropping them.


In [ ]:
from src.load_data import load_runs, print_quality_report

runs, report = load_runs()
print_quality_report(report)
print()
display_cols = ["start_date", "name", "distance_mi", "pace_min_per_mi", "average_hr", "pace_outlier", "use_for_fitness"]
runs[display_cols].head(10)


## 1. Target pace translation

Convert the BQ standard and safe target into required marathon pace, then supporting training paces via **Jack Daniels VDOT** continuous formulas (not a opaque lookup table).


In [ ]:
from src.pace_models import describe_target_paces
import pprint

paces = describe_target_paces(target.standard_seconds, target.safe_target_seconds)
pprint.pp(paces)


## 2. Current fitness over time (Riegel)

For each week, estimate equivalent marathon fitness from the best recent efforts in a rolling ~4-week window using Riegel:

\[ T_2 = T_1 \times (D_2 / D_1)^{1.06} \]

This yields a **time series** of predicted marathon finish times — not a single gauge.


In [ ]:
from src.fitness_trend import weekly_fitness_series, per_run_marathon_equiv
from src.pace_models import riegel_predict
from src.bq_standards import MARATHON_METERS

# Hand-check: longest run
fit = runs[runs["use_for_fitness"]]
longest = fit.loc[fit["distance_m"].idxmax()]
eq = riegel_predict(float(longest.moving_time_s), float(longest.distance_m))
print("Hand-check — longest run:")
print(f"  {longest.start_date.date()} | {longest.name} | {longest.distance_m/1000:.2f} km in {format_hms(longest.moving_time_s)}")
print(f"  Riegel marathon equivalent: {format_hms(eq)}")
print(f"  Manual: T * (42195/d)^1.06 = {longest.moving_time_s:.1f} * ({MARATHON_METERS}/{longest.distance_m:.1f})^1.06 = {format_hms(eq)}")

series = weekly_fitness_series(runs)
series.assign(pred=series["pred_marathon_s"].map(format_hms))[["week_start", "pred", "n_efforts", "best_source"]].tail(12)


## 3. Gap-to-BQ trend and projection

Plot predicted marathon time against the BQ standard and safe target. Fit a trend on recent weeks and ask when (if ever) it crosses the targets. Flat or worsening trends are stated plainly — no hopeful extrapolation.


In [ ]:
import matplotlib.pyplot as plt
from src.fitness_trend import project_gap, feasibility

proj = project_gap(
    series,
    target.standard_seconds,
    target.safe_target_seconds,
    recent_weeks=14,
    horizon_date=CONFIG.qualifying_race_date or CONFIG.qualifying_window_end,
)
print(proj.message)
print()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(series["week_start"], series["pred_marathon_s"]/3600, color="#1c3d5a", marker="o", lw=2, label="Predicted marathon fitness")
ax.axhline(target.standard_seconds/3600, color="#8a3b12", ls="--", label=f"BQ standard {format_hms(target.standard_seconds)}")
ax.axhline(target.safe_target_seconds/3600, color="#1f6b4a", ls="--", label=f"Safe target {format_hms(target.safe_target_seconds)}")
ax.set_ylabel("Hours")
ax.set_title("Equivalent marathon fitness vs BQ targets")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 4. Diagnostics — why the trend looks like this

Mileage, long-run progression, and gaps >7 days contextualize the fitness series. HR-based aerobic efficiency is included only when HR data exists (it does not in the current export).


In [ ]:
from src.diagnostics import build_diagnostics

diag = build_diagnostics(runs)
print(diag.narrative())

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(diag.weekly["week_start"], diag.weekly["miles"], width=5, color="#2f5d50", alpha=0.75)
ax.set_ylabel("Weekly miles")
ax.set_title("Weekly mileage (context for fitness stalls)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 5. Race-day feasibility verdict


In [ ]:
horizon = CONFIG.qualifying_race_date or CONFIG.qualifying_window_end
print(feasibility(series, proj, target.standard_seconds, target.safe_target_seconds, horizon))


## Caveats

- Riegel from easy short runs **overestimates** marathon time pessimistically if those runs were not race-effort; it can also be optimistic if GPS pauses inflate moving time. Treat the series as a trend signal, not a race prediction.
- No HR streams → no interval detection; all features come from per-activity aggregates.
- Age in `config.py` must be correct for the standard to be right.
- Acceptance is not the published standard alone — that is why the safe buffer exists.
